# Sentinel-3 / SLSTR (Sea and Land Surface Temperature Radiometer)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=sentinel/sentinel-3-slstr). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1

Level 1 (RBT): calibrated radiances and brightness temperatures, geolocated, for the nadir and oblique views.

Searches the Copernicus Data Space catalogue (anonymous). Set DOWNLOAD = True to fetch the first product.

In [ ]:
%pip install -q requests

In [ ]:
import requests

CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
AREA = "POLYGON((12.0 41.0, 13.0 41.0, 13.0 42.5, 12.0 42.5, 12.0 41.0))"   # Italy (lon lat)
START, END = "2024-06-01T00:00:00.000Z", "2024-06-03T00:00:00.000Z"

flt = (
    f"Collection/Name eq 'SENTINEL-3' "
    f"and Attributes/OData.CSC.StringAttribute/any(a:a/Name eq 'productType' "
    f"and a/OData.CSC.StringAttribute/Value eq 'SL_1_RBT___') "
    f"and ContentDate/Start gt {START} and ContentDate/Start lt {END} "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;{AREA}')"
)
response = requests.get(CATALOGUE, params={"$filter": flt, "$orderby": "ContentDate/Start asc", "$top": 5})
response.raise_for_status()
products = response.json()["value"]
print(len(products), "SLSTR RBT products found")
for p in products:
    print(p["Name"], round(p["ContentLength"] / 1e6), "MB")

# Searching is anonymous. Downloading needs a free Copernicus Data Space account.
DOWNLOAD = False
if DOWNLOAD and products:
    import getpass
    token = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={"client_id": "cdse-public", "grant_type": "password",
              "username": input("CDSE e-mail: "), "password": getpass.getpass("CDSE password: ")},
    ).json()["access_token"]
    product = products[0]
    url = "https://download.dataspace.copernicus.eu/odata/v1/Products(%s)/$value" % product["Id"]
    with requests.get(url, headers={"Authorization": "Bearer " + token}, stream=True) as r:
        r.raise_for_status()
        with open(product["Name"] + ".zip", "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print("saved", product["Name"] + ".zip")

# After downloading and unzipping, open the product with satpy, for example:
#   from glob import glob
#   from satpy import Scene
#   scn = Scene(filenames=glob("S3*.SEN3/*.nc"), reader="olci_l1b")   # or "slstr_l1b"
#   scn.load(["Oa08"]); scn["Oa08"].plot()

## L2

Level 2: land surface temperature (LST), sea surface temperature (WST), fire radiative power (FRP) and aerosol optical depth (AOD).

Downloads the 1 km land surface temperature layer of one SLSTR LST product from the Planetary Computer (no account needed) and plots it in swath geometry.

In [ ]:
%pip install -q pystac-client planetary-computer xarray netCDF4 requests matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import requests
import xarray as xr

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)



search = catalog.search(
    collections=["sentinel-3-slstr-lst-l2-netcdf"],
    bbox=[12.4, 41.8, 12.6, 42.0],                      # Rome - change to your area
    datetime="2024-06-01/2024-06-05",
)
items = search.item_collection()
if not items:
    raise SystemExit("No products found - widen the dates or the bounding box")
item = items[0]
print(len(items), "products; using", item.id)

with requests.get(item.assets["lst-in"].href, stream=True) as r:
    r.raise_for_status()
    with open("lst.nc", "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)

ds = xr.open_dataset("lst.nc")
print(ds)
lst = ds["LST"] if "LST" in ds else ds[list(ds.data_vars)[0]]
lst.plot(cmap="inferno")                   # kelvin, swath geometry
plt.title(item.id)
plt.show()